# Algoritmo de Construcción Iterativa de Modelos mediante Regresión Simbólica (PySR)

Este notebook implementa la **versión 0.5** del algoritmo iterativo para encontrar una ecuación matemática que describa la **Energía de Enlace Nuclear (BE)**, comparándolo con el modelo teórico **MISR**.

> 📁 **Nota:** este notebook es la versión **didáctica y autocontenida** del algoritmo (ejecuta su propio código paso a paso). El mismo algoritmo, ya modularizado en dos etapas (`model.py` → CSV + metadata, `plots.py` → figuras), vive en `../src/`.

**Cambios respecto a versiones anteriores:**
- 🟢 **v0.2**: el conjunto de **entrenamiento se restringe a núcleos con Z ≤ 50**; la comparación final se hace sobre el dataset completo. Se añadieron `cbrt` e `inv` como operadores unarios.
- 🟢 **v0.3**: **selección híbrida de variables** combinando *Información Mutua* (`mutual_info_regression`) y *Gradient Boosting* (BDT). Se reincorporó la variable **P** (factor de Casten) al conjunto de features.
- 🟢 **v0.4**: **restricción física suave** $BE(Z=0)=0$ mediante *puntos ancla* sintéticos, y **composición de operadores unarios** (`constraints=3`) para permitir potencias fraccionarias ($A^{2/3}$, $A^{-1/3}$, …) sin forzarlas.
- 🟢 **v0.5 — modo MISR**: configuración orientada a **acercarse a la ecuación MISR** (solo parámetros/modelo, sin forzar la estructura): `s=7` (incluye **I**), `n_t=45`, `niterations=80`, `maxdepth=12`, más poblaciones, operadores unarios `square/cbrt/inv` (**sin sqrt**) y **anclas desactivadas** (`anchor_z0=False`), porque MISR diverge en Z=0 y las anclas lo impedirían.

---

## Pseudocódigo del Algoritmo

```
CONSTRUCCIÓN ITERATIVA DEL MODELO:
  Mientras iteración ≤ max_iter Y ratio_mejora ≥ θ:
    1. Realizar división de datos en k grupos (k-fold) sobre el conjunto de entrenamiento (Z ≤ 50).
    2. Para cada grupo (fold):
       a. Evaluar la importancia de las variables combinando Información Mutua (MI) y
          Árboles de Decisión Potenciados (BDT), y seleccionar un subconjunto X_sub de tamaño s.
       b. (Opcional) Inyectar anclas físicas en Z=0 (BE=0) — DESACTIVADO en v0.5 (modo MISR).
       c. Realizar Regresión Simbólica (SR) con n_t términos en X_sub para predecir Y.
       d. Almacenar las ecuaciones de mejor rendimiento basadas en evaluación multiobjetivo.
    3. Seleccionar la mejor ecuación de entre todos los grupos.
    4. Actualizar Y para que sean los residuos del mejor modelo actual.
    5. Incrementar iteración.

FINALIZACIÓN DEL MODELO:
  Sumar las ecuaciones de cada iteración para formar el modelo final y propagar incertidumbres.

COMPARACIÓN:
  Evaluar el modelo PySR y la ecuación MISR sobre el dataset COMPLETO (todos los Z).
```

---

## Resumen de Implementación vs. Algoritmo

| Paso del Algoritmo | Estado | Comentario |
|---|---|---|
| División k-fold sobre entrenamiento (Z ≤ 50) | ✅ Implementado | `KFold` con `n_splits=5` sobre núcleos Z ≤ 50 |
| Árboles de Decisión Potenciados (BDT) para importancia | ✅ Implementado | `GradientBoostingRegressor` |
| Regresión de Información Mutua para importancia | ✅ **Implementado (v0.3)** | `mutual_info_regression` + selección híbrida normalizada |
| Seleccionar subconjunto X_sub de tamaño s | ✅ Implementado | `s=7` en v0.5 (incluye I, que MISR necesita) |
| Restricción física BE(Z=0)=0 | ⚙️ **Desactivada (v0.5)** | Anclas en conflicto con MISR (diverge en Z=0) |
| Regresión Simbólica con n_t términos | ✅ Implementado | PySR con `maxsize=45` |
| Evaluación multiobjetivo | ⚠️ Parcial | Solo primer término de la pérdida |
| Seleccionar mejor ecuación entre folds | ✅ Implementado |  |
| Actualizar Y a residuos | ✅ Implementado | `y = y - y_pred_global` |
| Condición de parada (ratio_mejora ≥ θ) | ✅ Implementado | `theta=0.04` |
| Sumar ecuaciones para modelo final | ✅ Implementado | Suma simbólica con sympy |
| Propagación de incertidumbres | ❌ **No implementado** | Falta propagación formal de errores |
| Optimización de hiperparámetros con validación | ❌ **No implementado** | Parámetros fijados manualmente |


---

## 0. Importación de Librerías

Se importan las librerías necesarias:

- **`pandas`**: manipulación de datos tabulares.
- **`numpy`**: cálculo numérico vectorizado.
- **`pathlib.Path`**: construcción de rutas de archivo multiplataforma.
- **`sklearn.model_selection.KFold`**: validación cruzada k-fold (paso 1 del algoritmo).
- **`sklearn.ensemble.GradientBoostingRegressor`**: BDT para evaluar importancia de variables (paso 2a).
- **`sklearn.feature_selection.mutual_info_regression`**: Información Mutua para la selección híbrida *(v0.3)*.
- **`pysr.PySRRegressor`**: motor de regresión simbólica (paso 2c).
- **`matplotlib.pyplot`**: visualización de resultados.
- **`sympy`**: manipulación simbólica para simplificar la ecuación final.


In [ ]:
"""
ALGORITMO DE CONSTRUCCIÓN ITERATIVA DE MODELOS MEDIANTE REGRESIÓN SIMBÓLICA (PySR)
Propósito: Encontrar una ecuación matemática que describa la Energía de Enlace (BE)
comparándolo con el modelo teórico MISR. Versión: 0.5

Versión 0.2: el set de entrenamiento se redujo solo para Z <= 50.
Versión 0.3: selección híbrida de variables (Información Mutua + Gradient Boosting); se
             reincorporó P (factor de Casten).
Versión 0.4: restricción física suave BE(Z=0) = 0 mediante puntos ancla sintéticos y
             composición de operadores unarios (constraints=3).
Versión 0.5: modo MISR — configuración (solo parámetros) para acercarse a la ecuación MISR:
             s=7 (incluye I), n_t=45, niterations=80, maxdepth=12, más poblaciones,
             unarios square/cbrt/inv (sin sqrt) y anclas desactivadas (MISR diverge en Z=0).
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_selection import mutual_info_regression
from pysr import PySRRegressor
import matplotlib.pyplot as plt
import sympy as sp
import tempfile
from pathlib import Path

---

## 1. Carga y Preprocesamiento de Datos

Este paso **no aparece explícitamente en el pseudocódigo**, pero es un prerrequisito fundamental.

### ¿Qué se hace aquí?

1. **Carga del CSV** `../datasets/be_exp.csv` con datos experimentales de núcleos atómicos (Z, N, BE, etc.).
2. **Limpieza**: Conversión de columnas a numérico, eliminación de filas con `NaN`.
3. **Filtro por Z ≤ 50** *(v0.2)*: Se separa el dataset completo (`dataset_full`) del dataset de entrenamiento (`dataset`) restringido a núcleos ligeros.
4. **Feature Engineering** — Variables físicas derivadas:
   - **A** = Z + N → Número de masa total
   - **BEpA** = BE / A → Energía de enlace por nucleón
   - **I** = (N − Z) / A → Isospin relativo
   - **P** = (nn × np) / (nn + np) → Factor de Casten *(reincorporado como feature en v0.3)*
5. **Definición de X e Y**:
   - **X** (características): `Z, N, A, P, I, nn, np` — 7 variables *(P incluida desde v0.3)*
   - **Y** (variable objetivo): `BE`
6. **Pesos**: $w_i = \frac{1}{1 + \sigma_i}$ — inversamente proporcionales a la incertidumbre experimental.

> ⚠️ **Diferencia clave con v0.1**: el entrenamiento se realiza **exclusivamente sobre núcleos con Z ≤ 50**, mientras que la evaluación final se hace sobre el dataset completo (`dataset_full`). Esto permite evaluar la capacidad de **extrapolación** del modelo hacia núcleos pesados.


In [ ]:
# =============================================================================
# 1. CARGA Y PREPROCESAMIENTO DE DATOS
# =============================================================================
# El CSV vive en ../datasets/ relativo a este notebook.
directorio_notebook = Path(".").resolve()
ruta_dataset = directorio_notebook.parent / "datasets" / "be_exp.csv"

dataset = pd.read_csv(ruta_dataset)

# Limpieza: Asegurar que las columnas sean numéricas y eliminar valores nulos.
for col in dataset.columns:
    if col != 'spinAndParity':
        dataset[col] = pd.to_numeric(dataset[col], errors='coerce')

dataset = dataset.dropna()

# =============================================================================
# FILTRO: Entrenar solo con núcleos Z <= 50
# =============================================================================
dataset_full = dataset.copy()          # Copia completa para comparación final
dataset = dataset[dataset['Z'] <= 50].reset_index(drop=True)
print(f"\n[FILTRO] Entrenamiento restringido a Z <= 50: {len(dataset)} núcleos (de {len(dataset_full)} totales)")

# Creación de variables físicas para el dataset COMPLETO
dataset_full['A']    = dataset_full['Z'] + dataset_full['N']
dataset_full['BEpA'] = dataset_full['BE'] / dataset_full['A']
dataset_full['I']    = (dataset_full['N'] - dataset_full['Z']) / dataset_full['A']
dataset_full['P']    = (dataset_full['nn'] * dataset_full['np']) / (dataset_full['nn'] + dataset_full['np'])

# Creación de variables físicas para el dataset de entrenamiento (Z <= 50)
dataset['A']    = dataset['Z'] + dataset['N']
dataset['BEpA'] = dataset['BE'] / dataset['A']
dataset['I']    = (dataset['N'] - dataset['Z']) / dataset['A']
dataset['P']    = (dataset['nn'] * dataset['np']) / (dataset['nn'] + dataset['np'])

print("\nDataset de entrenamiento con nuevas variables (A, BEpA, I, P):")
print(dataset[['Z', 'N', 'A', 'BE', 'BEpA', 'I', 'nn', 'np', 'P']].head())

---

## 2. Definición de Variables y Configuración de Hiperparámetros

Todo el conjunto Z ≤ 50 se usa para el bucle iterativo con K-Fold interno. La evaluación externa se realiza sobre `dataset_full`.

### Hiperparámetros de PySR (v0.5 — modo MISR)

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| `n_splits` (k) | 5 | Número de grupos para K-Fold |
| `s` | **7** | Tamaño del subconjunto de variables (todas, para incluir I) |
| `n_t` | **45** | Complejidad máxima SR (maxsize); MISR ≈ 30 nodos |
| `max_iter` | 1 | Iteraciones del bucle while (MISR es un único producto) |
| `theta` (θ) | 0.04 | Ratio de mejora mínimo para continuar |
| `niterations` | **80** | Iteraciones internas de PySR |
| `ncycles_per_iter` | 2000 | Ciclos por iteración PySR |
| `maxdepth` | **12** | Profundidad máxima del árbol simbólico |
| `populations` | **40** | Nº de poblaciones (más exploración) |
| `population_size` | **50** | Tamaño de cada población |
| `parsimony` | 0.0001 | Penalización por complejidad |
| `binary_ops` | +, −, ×, ÷ | Operadores binarios básicos |
| `unary_ops` | **square, cbrt, inv** | Vocabulario de MISR (**sin sqrt**) |
| `constraints` (unarios) | 3 | Complejidad máx. del argumento → permite `inv(square(Z))=N/Z²`, `cbrt(A)` |
| `anchor_z0` | **False** | Anclas BE(Z=0)=0 **desactivadas** (conflicto con MISR) |

### Modo MISR *(v0.5)*: acercarse a la ecuación teórica

Objetivo: que la SR se acerque a
$$BE_{MISR} = \eta_0 Z\left(1 + \tfrac{1}{N} - \tfrac{aN}{Z^2}\right)\left[I\left(b - \tfrac{A^{1/3}N}{Z}\right) + c\right]$$
**solo configurando parámetros** (sin forzar la estructura). Claves:

- **Variables**: MISR usa `Z, N, A, I`. Como **I es la variable peor rankeada** por la importancia híbrida (puesto 7), la única forma de incluirla manteniendo las 7 features es `s=7`.
- **Operadores**: MISR necesita `inv` (1/N, 1/Z²), `square` (Z²) y `cbrt` (A^{1/3}); **no usa `sqrt`**, por eso se quita (enfoca la búsqueda; no fuerza la ecuación).
- **Tamaño/depth/búsqueda**: la expresión MISR tiene ~30 nodos → `n_t=45`, `maxdepth=12`, y más `niterations`/poblaciones para hallar una estructura compleja.
- **Un solo producto**: MISR no es aditiva → `max_iter=1` (el bucle de residuos construiría una *suma*).
- **Anclas off**: MISR diverge en Z=0 (`-aN/Z²`) y N=0 (`1/N`); las anclas BE(Z=0)=0 penalizarían justo esos términos, así que se **desactivan**. (Se pierde la garantía BE(Z=0)=0.)

> ⚠️ **No hay garantía** de recuperar MISR exacta (la SR es estocástica); el objetivo es *acercarse* estructuralmente.


In [ ]:
# =============================================================================
# 2. DEFINICIÓN DE VARIABLES Y CONFIGURACIÓN DE HIPERPARÁMETROS
# =============================================================================

# Definición de características (X) y variable objetivo (y)
# v0.3+: se reincorpora P (factor de Casten) -> 7 variables.
X = dataset[['Z', 'N', 'A', 'P', 'I', 'nn', 'np']]
y = dataset['BE']

# Marca de variables discretas (mejora la precisión de la Información Mutua).
# Orden de columnas de X: ['Z', 'N', 'A', 'P', 'I', 'nn', 'np']
is_discrete = [True, True, True, False, False, True, True]

# Sigma experimental → Pesos
sigma_exp = dataset['bindingEnergyUncertainty'] if 'bindingEnergyUncertainty' in dataset.columns else 0.0
pesos = 1.0 / (1.0 + sigma_exp)

# Configuración del K-Fold (todo el set Z <= 50 con validación cruzada interna).
n_splits = 5
X_train_cv     = X.reset_index(drop=True)
y_train_cv     = y.reset_index(drop=True)
pesos_train_cv = pesos.reset_index(drop=True)

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# v0.5 — Configuración para ACERCARSE a la ecuación MISR (solo parámetros/modelo):
#   MISR usa Z,N,A,I; I es la variable peor rankeada (puesto 7), así que s=7 es la
#   única forma de incluirla manteniendo las 7 features. La expresión MISR tiene
#   ~30 nodos (n_t alto), es un único producto (max_iter=1) y usa inv/square/cbrt
#   (sin sqrt).
s   = 7    # usar TODAS las features para que I (rank 7) entre en la selección
n_t = 45   # complejidad máxima en SR (la expresión MISR tiene ~30 nodos)

# ── Parámetros de PySRRegressor centralizados (v0.5) ─────────────────────
niterations       = 80
ncycles_per_iter  = 2000
maxdepth          = 12
parsimony         = 0.0001
random_state_pysr = 11
populations       = 40
population_size   = 50
binary_ops        = ["+", "-", "*", "/"]
unary_ops         = ["square", "cbrt", "inv"]   # vocabulario de MISR (sin sqrt)

# ── Restricción física suave: BE(Z=0) = 0 ────────────────────────────────
# Desactivada en v0.5: las anclas penalizan los términos divergentes de MISR
# (1/N, N/Z²), así que entran en conflicto con acercarse a MISR. Ya NO se
# garantiza BE(Z=0)=0.
anchor_z0     = False                     # restricción suave desactivada (modo MISR)
anchor_weight = 10.0                      # peso por punto ancla (si se reactivara)
anchor_n_grid = list(range(0, 181, 10))   # barrido de N para las anclas


def make_anchors(selected_vars, n_grid=anchor_n_grid, weight=anchor_weight):
    """Construye filas ancla en Z=0 que imponen la restricción suave BE(Z=0)=0.

    Para cualquier subconjunto de ['Z','N','A','P','I','nn','np'] los valores se hacen
    físicamente consistentes en Z=0 (A=N, I=1, P=0, nn=np=0). Devuelve el DataFrame de
    features (ya renombrado N->Neut, I->Iso), el objetivo (ceros) y los pesos.
    """
    rows = []
    for n in n_grid:
        full = {
            'Z': 0.0,
            'N': float(n),
            'A': float(n),          # A = Z + N = N
            'I': 1.0,               # (N - Z)/A = N/N = 1   (N > 0)
            'P': 0.0,               # sin protones de valencia (np = 0) -> P = 0
            'nn': 0.0, 'np': 0.0,   # referencia sin protones/neutrones de valencia
        }
        rows.append({v: full[v] for v in selected_vars})
    Xa = pd.DataFrame(rows).rename(columns={'N': 'Neut', 'I': 'Iso'})
    ya = pd.Series([0.0] * len(n_grid))            # objetivo BE = 0 en Z = 0
    wa = pd.Series([float(weight)] * len(n_grid))  # peso alto para forzarlo
    return Xa, ya, wa


# Variables del bucle while principal
max_iter     = 1
theta        = 0.04
iteracion    = 1
ratio_mejora = 1.0
mse_anterior = float('inf')

modelos_finales = []  # Almacena la ecuación ganadora de cada iteración

print("Configuración completada.")
print(f"  Dataset entrenamiento: {len(X)} núcleos (Z <= 50)")
print(f"  Variables (X): {list(X.columns)}")
print(f"  s={s}, n_t={n_t}, max_iter={max_iter}, theta={theta}")
print(f"  PySR: niterations={niterations}, ncycles={ncycles_per_iter}, maxdepth={maxdepth}, "
      f"populations={populations}, population_size={population_size}")
print(f"  Operadores unarios: {unary_ops}")
print(f"  Restricción BE(Z=0)=0: anchor_z0={anchor_z0} (modo MISR)")

---

## 3. Bucle Principal — Construcción Iterativa del Modelo

Este es el **núcleo del algoritmo**. Corresponde directamente al bloque `while` del pseudocódigo.

### Flujo de cada iteración

```
┌──────────────────────────────────────────────────────────┐
│  ITERACIÓN i                                             │
│                                                          │
│  Para cada fold k:                                       │
│    ┌────────────────────────────────────────────┐        │
│    │ 1. MI + GradientBoosting → Importancia      │ ← MI+BDT│
│    │    híbrida (promedio normalizado)           │        │
│    │ 2. Seleccionar top-s variables (s=7)        │        │
│    │ 3. [anclas Z=0: DESACTIVADO en v0.5]        │        │
│    │ 4. PySR en X_sub → Ecuación simbólica       │ ← SR   │
│    │ 5. Evaluar MSE en validación (datos reales) │        │
│    └────────────────────────────────────────────┘        │
│                                                          │
│  Seleccionar mejor ecuación (menor MSE global)           │
│  Y ← Y - predicción(mejor_modelo)  [residuos]            │
│  Calcular ratio_mejora                                   │
└──────────────────────────────────────────────────────────┘
```

### Detalles importantes de la implementación

**Paso 1 — Selección híbrida de variables *(v0.3)*:**
- ✅ `mutual_info_regression` (Información Mutua) con marca de variables discretas.
- ✅ `GradientBoostingRegressor` para `feature_importances_`.
- ✅ Ambas puntuaciones se **normalizan a [0,1]** y se **promedian**; se eligen las `s` mejores. Con `s=7` se usan todas (para incluir I, que MISR necesita).

**Paso 3 — Anclas físicas:** **desactivadas en v0.5** (`anchor_z0=False`), porque entran en conflicto con los términos divergentes de MISR (1/N, N/Z²).

**Paso 4 — Regresión Simbólica (PySR):**
- Operadores binarios: `+, -, *, /`
- Operadores unarios: `square, cbrt, inv` *(sin `sqrt` en v0.5 — vocabulario de MISR)*
- Restricciones: `square`, `cbrt`, `inv` no pueden anidarse a sí mismos; argumento de complejidad ≤ 3 (permite `inv(square(Z))=N/Z²`, `cbrt(A)=A^{1/3}`).
- Función de pérdida: `loss(prediction, target, weight) = (weight * (prediction - target))²`

> **Nota sobre renombrado de columnas**: PySR no acepta `N` e `I` como nombres de variable porque coinciden con símbolos reservados. Por eso `N` → `Neut` e `I` → `Iso`.


In [ ]:
# =============================================================================
# 3. BUCLE PRINCIPAL — CONSTRUCCIÓN ITERATIVA DEL MODELO
# =============================================================================

print(f"\nIniciando Construcción Iterativa del Modelo...")
while iteracion <= max_iter and ratio_mejora >= theta:
    print(f"\n==========================================")
    print(f" ITERACIÓN {iteracion}")
    print(f"==========================================")

    mejores_ecuaciones = []

    print(f"\nRealizando K-Folding y Regresión Simbólica:")
    for fold, (train_index, val_index) in enumerate(kf.split(X_train_cv)):
        X_train, X_val = X_train_cv.iloc[train_index], X_train_cv.iloc[val_index]
        y_train, y_val = y_train_cv.iloc[train_index], y_train_cv.iloc[val_index]
        pesos_train    = pesos_train_cv.iloc[train_index]

        # ── Paso 1: Selección híbrida de variables (MI + BDT) ────────────────
        # Score 1: Información Mutua (MI)
        mi_scores = mutual_info_regression(X_train, y_train, discrete_features=is_discrete, random_state=42)
        mi_s = pd.Series(mi_scores, index=X.columns)

        # Score 2: Gradient Boosting (BDT) para importancia predictiva
        gbr = GradientBoostingRegressor(random_state=42)
        gbr.fit(X_train, y_train)
        gbr_s = pd.Series(gbr.feature_importances_, index=X.columns)

        # Normalizar ambas métricas a [0,1] y promediar (mismo peso)
        mi_norm  = mi_s / mi_s.max() if mi_s.max() > 0 else mi_s
        gbr_norm = gbr_s / gbr_s.max() if gbr_s.max() > 0 else gbr_s
        importancia_hibrida = ((mi_norm + gbr_norm) / 2).sort_values(ascending=False)

        vars_seleccionadas = importancia_hibrida.head(s).index.tolist()
        X_sub_train = X_train[vars_seleccionadas]

        print(f"\n--- Fold {fold+1} ---")
        print(f"Variables seleccionadas: {vars_seleccionadas}")

        # ── Paso 4: Regresión Simbólica con PySR ─────────────────────────────
        X_sub_train_pysr = X_sub_train.rename(columns={'N': 'Neut', 'I': 'Iso'})
        X_unidades       = ["" for _ in vars_seleccionadas]

        model = PySRRegressor(
            # ── Operadores Disponibles ──────────────────
            binary_operators=binary_ops,
            unary_operators=unary_ops,

            # ── Parámetros de la Búsqueda ───────────────
            niterations=niterations,
            ncycles_per_iteration=ncycles_per_iter,
            maxsize=n_t,
            maxdepth=maxdepth,
            populations=populations,
            population_size=population_size,

            # ── Restricciones de Complejidad ────────────
            # Permite componer operadores unarios (p.ej. A^(2/3)=cbrt(square(A)),
            # A^(-1/3)=inv(cbrt(A))) SIN forzarlos; nested self=0 evita raíces raras.
            # (sqrt eliminado en v0.5: no forma parte del vocabulario de MISR.)
            constraints={
                "square": 3,
                "cbrt":   3,
                "inv":    3,
            },
            nested_constraints={
                "square": {"square": 0},
                "cbrt":   {"cbrt":   0},
                "inv":    {"inv":    0},
            },

            # ── Pérdida y Penalizaciones ────────────────
            elementwise_loss="loss(prediction, target, weight) = (weight * (prediction - target))^2",
            parsimony=parsimony,
            complexity_of_variables=1,
            complexity_of_constants=1,
            dimensional_constraint_penalty=10**5,

            # ── Configuración de Ejecución ──────────────
            random_state=random_state_pysr,
            deterministic=True,
            parallelism='serial',
            verbosity=0,
            progress=True,
            tempdir=tempfile.gettempdir(),
            delete_tempfiles=True,
        )

        # ── Paso 3: Inyectar anclas Z=0 SOLO en el entrenamiento de PySR ─────
        # (la selección de variables y la MSE de validación usan datos reales)
        if anchor_z0:
            Xa, ya, wa = make_anchors(vars_seleccionadas)
            X_fit = pd.concat([X_sub_train_pysr, Xa], ignore_index=True)
            y_fit = pd.concat([y_train.reset_index(drop=True), ya], ignore_index=True)
            w_fit = pd.concat([pesos_train.reset_index(drop=True), wa], ignore_index=True)
        else:
            X_fit, y_fit, w_fit = X_sub_train_pysr, y_train, pesos_train

        try:
            model.fit(
                X_fit,
                y_fit,
                weights=w_fit,
                X_units=X_unidades,
                y_units="Constants.MeV"
            )

            # Evaluación en validación interna (datos reales)
            X_sub_val_pysr = X_val[vars_seleccionadas].rename(columns={'N': 'Neut', 'I': 'Iso'})
            y_pred_val     = model.predict(X_sub_val_pysr)
            mse_val        = np.mean((y_val - y_pred_val)**2)

            mejor_ecuacion = str(model.sympy())
            mejores_ecuaciones.append({
                'fold':      fold + 1,
                'variables': vars_seleccionadas,
                'ecuacion':  mejor_ecuacion,
                'mse_val':   mse_val,
                'model':     model
            })
            print(f"Mejor Ecuación Fold {fold+1}: {mejor_ecuacion} (MSE Val: {mse_val:.4f})")
        except Exception as e:
            print(f"Error en Regresión Simbólica en Fold {fold+1}: {e}")

    # ── Resumen de la iteración ───────────────────────────────────────────
    print(f"\n--- Modelos de Mejor Rendimiento Almacenados en Iteración {iteracion} ---")
    for r in mejores_ecuaciones:
        print(f"Fold {r['fold']}: [Vars: {r['variables']}] -> {r['ecuacion']} (MSE Val: {r['mse_val']:.4f})")

    # ── Selección del mejor modelo global y actualización de Y ──────────
    if mejores_ecuaciones:
        mejor_resultado_global = min(mejores_ecuaciones, key=lambda x: x['mse_val'])
        print(f"\n[!] Mejor ecuación global: Fold {mejor_resultado_global['fold']} — MSE {mejor_resultado_global['mse_val']:.4f}")

        mejor_modelo = mejor_resultado_global['model']
        vars_optimas = mejor_resultado_global['variables']

        modelos_finales.append({
            'iteracion': iteracion,
            'ecuacion':  mejor_resultado_global['ecuacion'],
            'model':     mejor_modelo,
            'variables': vars_optimas,
            'mse_val':   mejor_resultado_global['mse_val']
        })

        # Ratio de mejora
        if mse_anterior != float('inf') and mse_anterior > 0:
            ratio_mejora = (mse_anterior - mejor_resultado_global['mse_val']) / mse_anterior
            print(f"Ratio de mejora respecto a iteración anterior: {ratio_mejora:.4f}")

        mse_anterior = mejor_resultado_global['mse_val']

        # Actualizar Y → residuos del mejor modelo actual
        X_global_sub = X[vars_optimas].rename(columns={'N': 'Neut', 'I': 'Iso'})
        y_pred_global = mejor_modelo.predict(X_global_sub)
        y = y - y_pred_global

        print("\nVariable Y actualizada a los residuos del mejor modelo.")
        print("Primeros 5 valores de los nuevos residuos (Y):")
        print(y.head())
    else:
        print("\n[!] No se encontraron ecuaciones válidas en esta iteración. Abortando.")
        break

    iteracion += 1

---

## 4. Finalización del Modelo

### Correspondencia con el Algoritmo

### ¿Qué se hace?

1. ✅ **Suma de ecuaciones**: Se recopilan las ecuaciones simbólicas (sympy) de cada iteración y se suman:
   $$\text{Modelo Final} = \sum_{i=1}^{n} f_i(X)$$

2. ✅ **Simplificación**: Se usa `sp.simplify()` y se redondean coeficientes a 2 decimales.

3. ❌ **Propagación de incertidumbres**: **No implementada**. El modelo final suma ecuaciones sin propagar errores formalmente.


In [ ]:
# =============================================================================
# 4. FINALIZACIÓN DEL MODELO
# =============================================================================
print("\n==========================================")
print(" FINALIZACIÓN DEL MODELO")
print("==========================================")
print("Ecuaciones seleccionadas en cada iteración:")

ecuaciones_sympy = []
for m in modelos_finales:
    print(f"Iteración {m['iteracion']}: {m['ecuacion']} (Vars: {m['variables']})")
    ecuaciones_sympy.append(m['model'].sympy())

try:
    ecuacion_total_sympy  = sum(ecuaciones_sympy)
    ecuacion_simplificada = sp.simplify(ecuacion_total_sympy)

    # Redondear coeficientes flotantes a 2 decimales
    ecuacion_redondeada = ecuacion_simplificada.xreplace(
        {n: sp.Float(round(float(n), 2))
         for n in ecuacion_simplificada.atoms(sp.Number)
         if isinstance(n, sp.Float)}
    )
    modelo_creado_simplificado_str = str(ecuacion_redondeada)

    # Generar versión LaTeX para el gráfico
    # Si la ecuación es Piecewise, extraer solo el caso general (el último)
    if isinstance(ecuacion_redondeada, sp.Piecewise):
        ecuacion_para_latex = ecuacion_redondeada.args[-1][0]
    else:
        ecuacion_para_latex = ecuacion_redondeada

    ecuacion_latex = sp.latex(ecuacion_para_latex)

    # Embellecer nombres de variables
    replacements = {"Neut": "N", "Iso": "I", "nn": "n_n", "np": "n_p"}
    for old, new in replacements.items():
        ecuacion_latex = ecuacion_latex.replace(old, new)

    print(f"\nModelo Final Sumado y Simplificado:\n{modelo_creado_simplificado_str}")
    print(f"\nEcuación LaTeX:\n{ecuacion_latex}")

except Exception as e:
    print(f"Error al simplificar con sympy: {e}")
    modelo_creado_simplificado_str = " + ".join([f"({m['ecuacion']})" for m in modelos_finales])
    ecuacion_latex = (
        modelo_creado_simplificado_str
        .replace("Neut", "N").replace("Iso", "I")
        .replace("nn", "n_n").replace("np", "n_p")
        .replace("*", "\\cdot ")
    )
    print(f"\nModelo Final Sumado:\n{modelo_creado_simplificado_str}")

---

## 5. Comparación con el Modelo MISR Teórico

Esta sección **no forma parte del algoritmo original**, sino que es una validación adicional.

### Diferencias clave respecto a v0.1

| Aspecto | v0.1 | v0.2 |
|---------|------|------|
| Dataset de comparación | Dataset de entrenamiento (mismo Z ≤ 50) | **Dataset completo** (todos los Z) |
| Métrica | MSE | **MAE** |

### Ecuación MISR (v0.2)

$$BE_{MISR} = \eta_0 Z \left(1 + \frac{1}{N} - \frac{a N}{Z^2}\right) \left[I \left(b - \frac{A^{1/3} N}{Z}\right) + c\right]$$

Con los parámetros:
- $a = 1.10$, $b = 32.43$, $c = 16.70$, $\eta_0 = 1.0 \text{MeV}$

### Métrica usada

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |BE_{real,i} - BE_{pred,i}|$$


In [ ]:
# =============================================================================
# 5. COMPARACIÓN CON ECUACIÓN MISR TEÓRICA
# =============================================================================
print("\n==========================================")
print(" COMPARACIÓN CON ECUACIÓN MISR TEÓRICA")
print("==========================================")

# Parámetros ajustados de MISR (v0.2)
a_misr  =  1.10
b_misr  = 32.43
c_misr  = 16.70
eta_0   =  1.0

# Variables del dataset COMPLETO (todos los Z)
Z_total  = dataset_full['Z']
N_total  = dataset_full['N']
A_total  = dataset_full['A']
I_total  = dataset_full['I']
BE_real  = dataset_full['BE']

# ── Cálculo de BE_MISR ────────────────────────────────────────────────────
# BE = eta_0 * Z * (1 + 1/N - a*N/Z^2) * [I*(b - A^{1/3}*N/Z) + c]
termino_par = 1 + 1/N_total - (a_misr * N_total) / (Z_total**2)
termino_bra = I_total * (b_misr - (A_total**(1/3) * N_total) / Z_total) + c_misr
BE_MISR     = eta_0 * Z_total * termino_par * termino_bra

# ── Predicción total del modelo iterativo PySR ───────────────────────────
BE_PySR = np.zeros(len(dataset_full))
for m in modelos_finales:
    vars_opt = m['variables']
    X_sub    = dataset_full[vars_opt].rename(columns={'N': 'Neut', 'I': 'Iso'})
    BE_PySR += m['model'].predict(X_sub)

# ── Métricas ─────────────────────────────────────────────────────────────
mae_pysr = np.mean(np.abs(BE_real - BE_PySR))
mae_misr = np.mean(np.abs(BE_real - BE_MISR))

print(f"MAE Modelo PySR (Total Iterativo): {mae_pysr:.4f} MeV")
print(f"MAE Ecuación MISR:                 {mae_misr:.4f} MeV")

---

## 6. Visualización — Gráfico Comparativo (4 Paneles)

### Cambios respecto a v0.1

En v0.1 se generaban **2 paneles**. En v0.2 se generan **4 paneles**:

1. **Predicción vs. Dato Experimental** — Scatter plot en el eje $BE_{exp}$ vs $BE_{pred}$. Los puntos ideales caen sobre la diagonal $y = x$.
2. **Residuos por Nucleón vs Z** *(nuevo)* — Muestra $\Delta BE / A$ en función del número atómico. La línea vertical en $Z = 50$ marca el límite del entrenamiento, permitiendo evaluar extrapolación.
3. **Residuos absolutos vs N** *(nuevo)* — Muestra $\Delta BE$ en función del número de neutrones.
4. **Tabla de parámetros de configuración** *(nuevo)* — Resumen automático de todos los hiperparámetros usados en la ejecución.

Además, el **título principal** ahora muestra la ecuación en formato LaTeX usando el motor MathText de Matplotlib (compatible sin LaTeX instalado).


In [ ]:
# =============================================================================
# 6. VISUALIZACIÓN — GRÁFICO COMPARATIVO (4 PANELES)
# =============================================================================
print("\nGenerando gráfico comparativo (4 paneles)...")

plt.rcParams.update({
    "text.usetex":       False,
    "font.family":       "serif",
    "mathtext.fontset":  "cm",
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "legend.fontsize":    9,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
})

fig = plt.figure(figsize=(10, 24))
plt.subplots_adjust(top=0.93, hspace=0.40)

# Título con ecuación en LaTeX
try:
    plt.suptitle(
        r"Ecuación PySR Total Simplificada:" + "\n" + rf"$BE = {ecuacion_latex}$",
        fontsize=14, color='darkblue', fontweight='bold'
    )
except Exception:
    plt.suptitle(
        f"Ecuación PySR Total Simplificada:\nBE = {modelo_creado_simplificado_str[:100]}...",
        fontsize=12, color='darkblue', fontweight='bold'
    )

# ── Panel 1: Predicciones vs Reales ──────────────────────────────────────
ax1 = plt.subplot(4, 1, 1)
ax1.scatter(BE_real, BE_PySR, alpha=0.8, label=r'PySR (Iterativo)', color='blue',  s=5)
ax1.scatter(BE_real, BE_MISR, alpha=0.8, label=r'MISR (Teórica)',   color='red',   marker='x', s=5)

min_val = min(BE_real.min(), BE_PySR.min(), BE_MISR.min())
max_val = max(BE_real.max(), BE_PySR.max(), BE_MISR.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.7, label=r'Valor Exacto')

ax1.set_xlabel(r'$BE_{\mathrm{exp}}$ (MeV)')
ax1.set_ylabel(r'$BE_{\mathrm{pred}}$ (MeV)')
ax1.set_title(r'Predicción vs Dato Experimental')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.5)

# ── Panel 2: Residuos por nucleón vs Z ───────────────────────────────────
ax2 = plt.subplot(4, 1, 2)
residuos_pysr_pA = (BE_PySR - BE_real) / A_total
residuos_misr_pA = (BE_MISR - BE_real) / A_total

ax2.scatter(Z_total, residuos_pysr_pA, alpha=0.8, label=rf'PySR   (MAE: {mae_pysr:.2f})', color='blue', s=5)
ax2.scatter(Z_total, residuos_misr_pA, alpha=0.8, label=rf'MISR   (MAE: {mae_misr:.2f})', color='red',  marker='x', s=5)
ax2.axhline(0, color='k', linestyle='--', alpha=0.7)
ax2.axvline(50, color='gray', linestyle=':', linewidth=1.5, alpha=0.8,
            label=r'$Z = 50$ (límite entrenamiento)')

ax2.set_xlim(10, 120)
ax2.set_xlabel(r'Número Atómico $Z$')
ax2.set_ylabel(r'$\Delta BE / A$ (MeV)')
ax2.set_title(r'Residuos por Nucleón vs $Z$')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.5)

# ── Panel 3: Residuos absolutos vs N ─────────────────────────────────────
ax3 = plt.subplot(4, 1, 3)
residuos_pysr_N = BE_PySR - BE_real
residuos_misr_N = BE_MISR - BE_real

ax3.scatter(N_total, residuos_pysr_N, alpha=0.8, label=rf'PySR   (MAE: {mae_pysr:.2f})', color='blue', s=5)
ax3.scatter(N_total, residuos_misr_N, alpha=0.8, label=rf'MISR   (MAE: {mae_misr:.2f})', color='red',  marker='x', s=5)
ax3.axhline(0, color='k', linestyle='--', alpha=0.7)

ax3.set_xlim(10, 90)
ax3.set_xlabel(r'Número de Neutrones $N$')
ax3.set_ylabel(r'$\Delta BE$ (MeV)')
ax3.set_title(r'Residuos vs $N$')
ax3.legend()
ax3.grid(True, linestyle='--', alpha=0.5)

# ── Panel 4: Tabla de parámetros de configuración ────────────────────────
ax4 = plt.subplot(4, 1, 4)
ax4.axis('off')

config_params = [
    [r"Versión",                         r"0.5 (modo MISR)"],
    [r"$Z$ máx. entrenamiento",           r"$Z \leq 50$"],
    [r"Restricción física $BE(Z{=}0)=0$", r"desactivada"],
    [r"n_splits",                         str(n_splits)],
    [r"s  (vars. seleccionadas)",         str(s)],
    [r"n_t  (maxsize)",                   str(n_t)],
    [r"max_iter",                         str(max_iter)],
    [r"$\theta$  (ratio mejora mín.)",    str(theta)],
    [r"niterations",                      str(niterations)],
    [r"ncycles_per_iteration",            str(ncycles_per_iter)],
    [r"maxdepth",                         str(maxdepth)],
    [r"populations",                      str(populations)],
    [r"population\_size",                 str(population_size)],
    [r"parsimony",                        str(parsimony)],
    [r"random\_state",                    str(random_state_pysr)],
    [r"Operadores binarios",              ", ".join(binary_ops)],
    [r"Operadores unarios",               ", ".join(unary_ops)],
]

table = ax4.table(
    cellText=config_params,
    colLabels=[r"Parámetro", r"Valor"],
    cellLoc='left',
    loc='center',
    colWidths=[0.55, 0.35],
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.55)

for col in range(2):
    table[(0, col)].set_facecolor('#2c3e50')
    table[(0, col)].set_text_props(color='white', fontweight='bold')

for row in range(1, len(config_params) + 1):
    fc = '#eaf0f6' if row % 2 == 0 else 'white'
    for col in range(2):
        table[(row, col)].set_facecolor(fc)

ax4.set_title(r'Parámetros de Configuración', fontsize=12, pad=12)

plt.savefig('comparacion_misr_pysr_v05.png', dpi=300, bbox_inches='tight')
print("\n[OK] Gráfico exportado como 'comparacion_misr_pysr_v05.png'")
plt.show()

---

## 6.b Comportamiento en $Z=0$ (anclas desactivadas)

En **v0.5 (modo MISR) las anclas están desactivadas** (`anchor_z0=False`), así que **ya no se impone** $BE(Z=0)=0$. Esto es coherente con acercarse a MISR, que de hecho **diverge** en $Z=0$ (término $-aN/Z^2$) y en $N=0$ ($1/N$). La siguiente celda evalúa el modelo en $Z=0$ solo de forma **informativa**: puede dar valores grandes o no finitos (inf/nan), y eso es lo esperado en este modo.


In [ ]:
# =============================================================================
# 6.b COMPORTAMIENTO EN Z = 0  (informativo; anclas desactivadas en v0.5)
# =============================================================================
# Con anchor_z0=False ya NO se impone BE(Z=0)=0. Evaluamos en Z=0 solo para
# observar el comportamiento (puede divergir, como MISR). Cálculo robusto a inf/nan.
n_grid_test = anchor_n_grid
test_z0 = pd.DataFrame({
    'Z':  [0.0] * len(n_grid_test),
    'N':  [float(n) for n in n_grid_test],
    'A':  [float(n) for n in n_grid_test],   # A = N en Z=0
    'P':  [0.0] * len(n_grid_test),
    'I':  [1.0] * len(n_grid_test),
    'nn': [0.0] * len(n_grid_test),
    'np': [0.0] * len(n_grid_test),
})

BE_z0 = np.zeros(len(test_z0))
for m in modelos_finales:
    X_sub = test_z0[m['variables']].rename(columns={'N': 'Neut', 'I': 'Iso'})
    with np.errstate(divide='ignore', invalid='ignore'):
        BE_z0 = BE_z0 + np.asarray(m['model'].predict(X_sub), dtype=float)

finitos = BE_z0[np.isfinite(BE_z0)]
print("Comportamiento del modelo en Z=0 (anclas desactivadas, modo MISR):")
print(f"  N evaluados:        {n_grid_test}")
print(f"  valores no finitos: {int(np.sum(~np.isfinite(BE_z0)))} / {len(BE_z0)}")
if finitos.size:
    print(f"  max |BE(Z=0)| (finitos) = {np.max(np.abs(finitos)):.4e} MeV")
print("  [i] BE(Z=0)=0 ya NO se impone en v0.5; valores grandes/no finitos son esperados.")

---

## 7. Resumen Final: Estado de Implementación vs. Algoritmo

### ✅ Implementado hasta v0.5

1. **Filtro Z ≤ 50** para el entrenamiento, con comparación sobre el dataset completo *(v0.2)*.
2. **Selección híbrida de variables**: Información Mutua + Gradient Boosting *(v0.3)*.
3. **Composición de operadores unarios** (`constraints=3`): potencias fraccionarias sin forzarlas *(v0.4)*.
4. **Restricción física suave** $BE(Z=0)=0$ vía anclas *(v0.4)* — **desactivable** (off en v0.5).
5. **Modo MISR** *(v0.5)*: configuración (solo parámetros) para acercarse a la ecuación MISR:
   `s=7` (incluye I), `n_t=45`, `niterations=80`, `maxdepth=12`, más poblaciones, unarios
   `square/cbrt/inv` (sin sqrt) y anclas desactivadas (MISR diverge en Z=0).
6. **Centralización de parámetros** y tabla de configuración automática.
7. **Comparación con MISR** (MAE) y **4 paneles de visualización** con LaTeX.

### ❌ Pendiente (trabajo futuro)

1. **Evaluación multiobjetivo** completa para almacenamiento de ecuaciones (Paso 2d del algoritmo).
2. **Propagación formal de incertidumbres** (Finalización del modelo).
3. **Optimización automática de hiperparámetros** usando un conjunto de validación.

### ⚠️ Notas del modo MISR (v0.5)

- **No hay garantía** de recuperar MISR exacta: la SR es estocástica y busca el mejor ajuste,
  no una forma prescrita. El objetivo es *acercarse* estructuralmente (factor `Z`, `1/N`,
  `N/Z²`, `cbrt(A)`, uso de `I`).
- **`s=7` mete P, nn, np como ruido** (necesario para incluir I, que rankea último). Una
  alternativa más limpia sería restringir las features a `[Z, N, A, I]`.
- **Se pierde `BE(Z=0)=0`** (decisión consciente; MISR tampoco lo cumple).

> 🔗 **Versión modular:** el mismo algoritmo separado en `../src/model.py` (entrena → `results/MISRComparison_results.csv` + `results/run_metadata.json`) y `../src/plots.py` (lee esos archivos → figuras, incluida la combinada `MISRComparison.png`).
